In [7]:
# ==============================================================================
# UCS420: Cognitive Computing - Assignment 10
# Roll Number: 1024160019 (Odd Roll Number -> Diabetes Care Information Assistant)
# ==============================================================================

import warnings
import ipywidgets as widgets
from IPython.display import clear_output, display
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# Download required NLTK datasets (runs silently in Colab / Jupyter)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

# ------------------------------------------------------------------------------
# REQUIREMENT 1: Labelled Corpus (48 queries across 6 intent categories >= 40 & >= 5)
# ------------------------------------------------------------------------------
corpus = {
    "Symptoms": [
        "What are the common symptoms of diabetes?",
        "Why am I feeling thirsty all the time?",
        "Is frequent urination a sign of high blood sugar?",
        "I feel extremely tired and fatigued every day",
        "Why is my vision getting blurry lately?",
        "Unexplained weight loss and constant hunger symptoms",
        "Slow healing cuts and tingling in my hands and feet",
        "How do I know if I have diabetes symptoms?",
    ],
    "Diet_Nutrition": [
        "What foods should a diabetic patient eat?",
        "Create a healthy diet plan for blood sugar control",
        "Can I eat fruits and rice if I have diabetes?",
        "Which foods cause high blood sugar spikes?",
        "Best breakfast and dinner options for diabetics",
        "Are whole grains and green vegetables good for diabetes?",
        "How much sugar and carbohydrates can I consume daily?",
        "Suggest low glycemic index foods for diabetic patients",
    ],
    "Blood_Sugar_Monitoring": [
        "What is the normal fasting blood sugar level?",
        "How often should I check my blood glucose levels?",
        "What should my post meal sugar level be?",
        "Explain HbA1c test range for diabetes control",
        "How to use a glucometer at home properly?",
        "What is considered dangerously high blood sugar?",
        "When is the best time to test blood glucose?",
        "Normal random blood sugar reading range",
    ],
    "Medication_Insulin": [
        "How does insulin help control blood sugar?",
        "When should I take my diabetes medication?",
        "What are the side effects of metformin tablets?",
        "How to store insulin pens safely at home?",
        "What happens if I miss a dose of diabetes medicine?",
        "Difference between Type 1 insulin and Type 2 pills",
        "Where is the best place to inject insulin?",
        "Can diabetes be managed without insulin injections?",
    ],
    "Exercise_Lifestyle": [
        "What exercises are best for diabetic patients?",
        "How does walking affect blood glucose levels?",
        "Can yoga and weight training reduce blood sugar?",
        "How many minutes should a diabetic exercise daily?",
        "Precautions before working out with diabetes",
        "Does stress and poor sleep increase blood sugar?",
        "Healthy lifestyle habits to manage Type 2 diabetes",
        "Is brisk walking after meals good for sugar control?",
    ],
    "Complications_Prevention": [
        "What are the long term complications of diabetes?",
        "How to prevent diabetic neuropathy and nerve damage?",
        "Can high blood sugar affect my kidneys and eyes?",
        "Why is daily foot care important for diabetics?",
        "How to reduce heart disease risk with diabetes?",
        "What causes diabetic retinopathy and vision loss?",
        "How can I prevent diabetes complications early?",
        "Signs of kidney problems in diabetic patients",
    ],
}

print(
    f"Total Intents: {len(corpus)} | Total Queries:"
    f" {sum(len(v) for v in corpus.values())}"
)

# ------------------------------------------------------------------------------
# REQUIREMENT 2: Hierarchical Taxonomy (>= 4 major categories, >= 2 subcategories each)
# ------------------------------------------------------------------------------
taxonomy = {
    "Clinical_Assessment": {
        "Symptom_Recognition": [
            "Hyperglycemia_Signs",
            "Neuropathy_Indicators",
        ],
        "Diagnostic_Monitoring": [
            "Fasting_Postprandial_Glucose",
            "HbA1c_Tracking",
        ],
    },
    "Therapeutic_Management": {
        "Pharmacotherapy": ["Oral_Hypoglycemics", "Insulin_Therapy"],
        "Emergency_Intervention": [
            "Hypoglycemia_Protocol",
            "Ketoacidosis_Alert",
        ],
    },
    "Lifestyle_Modulation": {
        "Medical_Nutrition_Therapy": [
            "Low_GI_Diet",
            "Carbohydrate_Counting",
        ],
        "Physical_Activity": ["Aerobic_Exercise", "Resistance_Training"],
    },
    "Preventive_Care": {
        "Microvascular_Protection": ["Retinopathy_Screening", "Nephropathy_Care"],
        "Macrovascular_&_Foot_Care": [
            "Cardiovascular_Health",
            "Diabetic_Foot_Hygiene",
        ],
    },
}

# ------------------------------------------------------------------------------
# REQUIREMENT 3: Domain Ontology (>= 8 concepts and >= 10 relationships)
# ------------------------------------------------------------------------------
ontology_concepts = [
    "Patient",
    "Diabetes_Mellitus",
    "Blood_Glucose",
    "Insulin",
    "Pancreas",
    "Diet_Plan",
    "Physical_Exercise",
    "Glucometer",
    "HbA1c_Test",
    "Diabetic_Complications",
]

ontology_relationships = [
    ("Patient", "diagnosed_with", "Diabetes_Mellitus"),
    ("Pancreas", "produces", "Insulin"),
    ("Insulin", "regulates", "Blood_Glucose"),
    ("Patient", "monitors", "Blood_Glucose"),
    ("Glucometer", "measures", "Blood_Glucose"),
    ("HbA1c_Test", "evaluates_3month_average_of", "Blood_Glucose"),
    ("Diet_Plan", "stabilizes", "Blood_Glucose"),
    ("Physical_Exercise", "improves_sensitivity_to", "Insulin"),
    ("Diabetes_Mellitus", "requires_management_via", "Diet_Plan"),
    ("Diabetes_Mellitus", "treated_with", "Insulin"),
    ("Uncontrolled_Blood_Glucose", "leads_to", "Diabetic_Complications"),
    ("Patient", "undergoes_screening_for", "Diabetic_Complications"),
]

# ------------------------------------------------------------------------------
# REQUIREMENT 4: NLP Preprocessing (Lowercase, Tokenization, Stopwords, Lemmatization)
# ------------------------------------------------------------------------------
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
  # 1. Lowercase conversion
  text = text.lower()
  # 2. Tokenization (keeping alphabetic tokens)
  tokens = [word for word in nltk.word_tokenize(text) if word.isalpha()]
  # 3. Stopword removal & 4. Lemmatization
  cleaned_tokens = [
      lemmatizer.lemmatize(word) for word in tokens if word not in stop_words
  ]
  return " ".join(cleaned_tokens)


# Prepare dataset
X_raw, y_labels = [], []
for intent, queries in corpus.items():
  for q in queries:
    X_raw.append(preprocess_text(q))
    y_labels.append(intent)

# ------------------------------------------------------------------------------
# REQUIREMENT 5: TF-IDF Feature Extraction & Classifier Training
# ------------------------------------------------------------------------------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_raw, y_labels, test_size=0.25, random_state=42, stratify=y_labels
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

# Using Logistic Regression (provides well-calibrated probability scores for confidence thresholding)
classifier = LogisticRegression(C=5.0, max_iter=1000, random_state=42)
classifier.fit(X_train_tfidf, y_train)

# ------------------------------------------------------------------------------
# REQUIREMENT 6: Model Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
# ------------------------------------------------------------------------------
y_pred = classifier.predict(X_test_tfidf)
print("\n--- MODEL EVALUATION METRICS ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report (Precision, Recall, F1-Score):")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ------------------------------------------------------------------------------
# REQUIREMENT 7: Domain-Specific Knowledge Base
# ------------------------------------------------------------------------------
knowledge_base = {
    "Symptoms": (
        "Common diabetes symptoms include excessive thirst (polydipsia),"
        " frequent urination, unexplained weight loss, extreme fatigue,"
        " blurred vision, and slow-healing wounds. Consult a physician for a"
        " fasting blood glucose test."
    ),
    "Diet_Nutrition": (
        "Focus on low-glycemic-index foods: leafy greens, whole grains, legumes,"
        " nuts, and lean proteins. Limit refined sugars, white rice, and"
        " sugary beverages, and practice portion control."
    ),
    "Blood_Sugar_Monitoring": (
        "Target ranges for most adults with diabetes: Fasting blood sugar:"
        " 80–130 mg/dL; 2 hours post-meal: < 180 mg/dL; HbA1c: < 7.0%. Track"
        " readings regularly using a calibrated glucometer."
    ),
    "Medication_Insulin": (
        "Take prescribed oral medications (like Metformin) or insulin strictly"
        " on schedule. Store unopened insulin in a refrigerator (2°C–8°C) and"
        " never skip or double a dose without doctor guidance."
    ),
    "Exercise_Lifestyle": (
        "Aim for at least 150 minutes of moderate aerobic activity per week"
        " (such as 30 minutes of brisk walking daily) plus resistance training."
        " Stay hydrated and check blood sugar before and after workouts."
    ),
    "Complications_Prevention": (
        "Prevent long-term complications by maintaining target HbA1c, blood"
        " pressure, and cholesterol. Schedule annual eye (retina) exams, kidney"
        " function tests, and daily foot inspections."
    ),
}

# ------------------------------------------------------------------------------
# REQUIREMENT 8 & 9: Rule Engine (>= 5 Rules) + Integrated Cognitive Assistant
# ------------------------------------------------------------------------------
EMERGENCY_KEYWORDS = [
    "unconscious",
    "fainting",
    "fainted",
    "seizure",
    "chest pain",
    "confusion",
    "shortness of breath",
    "severe hypoglycemia",
    "ketoacidosis",
    "vomiting",
    "sugar below 50",
    "sugar over 400",
    "emergency",
    "dizzy and sweating",
]

UNSUPPORTED_KEYWORDS = [
    "cancer",
    "fracture",
    "bone",
    "covid",
    "asthma",
    "pregnancy",
    "migraine",
    "arthritis",
    "typhoid",
    "malaria",
    "dental",
    "skin rash",
]


def diabetes_cognitive_assistant(user_query):
  query_lower = user_query.lower().strip()

  # RULE 1: Empty / Invalid Input Handling
  if not query_lower or len(query_lower) < 2:
    return (
        "[Rule 1 - Invalid Input]: Please enter a valid question related to"
        " diabetes care."
    )

  # RULE 2: Emergency Escalation Rule
  if any(keyword in query_lower for keyword in EMERGENCY_KEYWORDS):
    return (
        "🚨 [Rule 2 - EMERGENCY ESCALATION]: Your symptoms indicate a potential"
        " medical emergency (such as severe hypoglycemia or diabetic"
        " ketoacidosis). Seek immediate emergency medical care or call emergency"
        " services right away!"
    )

  # RULE 3: Unsupported Healthcare Domain Handling
  if any(keyword in query_lower for keyword in UNSUPPORTED_KEYWORDS):
    return (
        "⚠️ [Rule 3 - Unsupported Healthcare Topic]: I am specialized strictly"
        " in Diabetes Care. For other medical conditions, please consult an"
        " appropriate healthcare specialist."
    )

  # NLP Processing & Intent Classification
  cleaned = preprocess_text(user_query)
  if not cleaned:
    return (
        "🤔 [Rule 4 - Low Confidence]: Could not extract meaningful medical"
        " terms. Please rephrase your diabetes question."
    )

  vec = vectorizer.transform([cleaned])
  probabilities = classifier.predict_proba(vec)[0]
  max_prob = max(probabilities)
  predicted_intent = classifier.classes_[probabilities.argmax()]

  # RULE 4: Low-Confidence Handling Rule
  if max_prob < 0.24:
    return (
        f"🤔 [Rule 4 - Low Confidence (Score: {max_prob:.2f})]: I am not"
        " confident I understood your question. Please ask specifically about"
        " diabetes symptoms, diet, blood sugar levels, medication, exercise, or"
        " complications."
    )

  response = knowledge_base[predicted_intent]

  # RULE 5: Contextual Advisory / Urgency Escalation Rule
  if any(
      w in query_lower for w in ["persistent", "always", "worse", "high", "low"]
  ):
    response += (
        "\n📌 [Rule 5 - Clinical Advisory]: Because you mentioned persistent or"
        " fluctuating readings, please log your blood sugar levels and consult"
        " your endocrinologist."
    )

  return (
      f"[Intent: {predicted_intent} | Confidence: {max_prob:.2f}]\n{response}"
  )


# ------------------------------------------------------------------------------
# MANDATORY TEST SUITE (5 Required Test Cases from Assignment Sheet)
# ------------------------------------------------------------------------------
print("\n" + "=" * 75)
print("AUTOMATED VERIFICATION OF ALL 5 ASSIGNMENT TEST CASES")
print("=" * 75)

test_cases = [
    # 1. A query belonging to each defined intent
    ("1a. Intent [Symptoms]", "Why am I having blurry vision and high thirst?"),
    ("1b. Intent [Diet_Nutrition]", "What breakfast should a diabetic eat?"),
    (
        "1c. Intent [Blood_Sugar_Monitoring]",
        "What is the normal HbA1c and fasting glucose range?",
    ),
    (
        "1d. Intent [Medication_Insulin]",
        "How should I store my insulin pens safely?",
    ),
    (
        "1e. Intent [Exercise_Lifestyle]",
        "Is brisk walking good for blood sugar control?",
    ),
    (
        "1f. Intent [Complications_Prevention]",
        "How can I protect my kidneys and nerves from diabetes?",
    ),
    # 2. A differently worded query
    (
        "2. Differently Worded Query",
        "Tell me good meal ideas to keep my glucose steady",
    ),
    # 3. An unsupported healthcare query
    (
        "3. Unsupported Healthcare Query",
        "What medicine should I take for a fractured bone and asthma?",
    ),
    # 4. A low-confidence query
    ("4. Low-Confidence Query", "What is the weather forecast for tomorrow?"),
    # 5. An emergency-related query
    (
        "5. Emergency-Related Query",
        "Patient is fainting with chest pain and sugar below 50",
    ),
]

for label, query in test_cases:
  print(f"\nTEST {label}")
  print(f"User Query : {query}")
  print(f"Assistant  : {diabetes_cognitive_assistant(query)}")

# ------------------------------------------------------------------------------
# REQUIREMENT 10: Interactive Interface in Google Colab (Multi-Query Chat UI)
# ------------------------------------------------------------------------------
print("\n" + "=" * 75)
print("INTERACTIVE DIABETES CARE COGNITIVE ASSISTANT (GOOGLE COLAB UI)")
print("=" * 75)

input_box = widgets.Text(
    value="",
    placeholder="Type your diabetes care question here and press Send...",
    description="Query:",
    layout=widgets.Layout(width="75%"),
)
send_button = widgets.Button(description="Send", button_style="primary")
clear_button = widgets.Button(description="Clear History", button_style="warning")
chat_output = widgets.Output()


def on_send_clicked(b):
  query = input_box.value
  if query.strip():
    reply = diabetes_cognitive_assistant(query)
    with chat_output:
      print(f"🧑 You: {query}")
      print(f"🤖 Assistant: {reply}\n" + "-" * 60)
    input_box.value = ""


def on_clear_clicked(b):
  with chat_output:
    clear_output()


send_button.on_click(on_send_clicked)
clear_button.on_click(on_clear_clicked)

display(widgets.HBox([input_box, send_button, clear_button]), chat_output)

Total Intents: 6 | Total Queries: 48

--- MODEL EVALUATION METRICS ---
Accuracy: 0.67

Classification Report (Precision, Recall, F1-Score):
                          precision    recall  f1-score   support

  Blood_Sugar_Monitoring       0.50      0.50      0.50         2
Complications_Prevention       0.67      1.00      0.80         2
          Diet_Nutrition       1.00      1.00      1.00         2
      Exercise_Lifestyle       0.00      0.00      0.00         2
      Medication_Insulin       0.50      0.50      0.50         2
                Symptoms       0.67      1.00      0.80         2

                accuracy                           0.67        12
               macro avg       0.56      0.67      0.60        12
            weighted avg       0.56      0.67      0.60        12

Confusion Matrix:
[[1 0 0 0 1 0]
 [0 2 0 0 0 0]
 [0 0 2 0 0 0]
 [1 1 0 0 0 0]
 [0 0 0 0 1 1]
 [0 0 0 0 0 2]]

AUTOMATED VERIFICATION OF ALL 5 ASSIGNMENT TEST CASES

TEST 1a. Intent [Symptoms]
User 

Output()